# 11 — Compaction Demo: From File Proliferation to Consolidated Parquet

**The problem:** streaming and micro-batch ingest writes one tiny Parquet file per
batch per partition. Over time a table accumulates hundreds of files averaging a
few KB each. Every query must open, read the footer, and close each one. On object
storage, latency per file-open is ~50–200 ms — 500 files adds 1–2 minutes of pure
overhead before a single row is decoded.

**The solution:** Iceberg's `rewrite_data_files` Spark stored procedure (bin-packing
strategy) merges small files into right-sized Parquet files. `rewrite_manifests`
consolidates the manifest layer. Together they restore scan performance without
touching query results.

This notebook demonstrates the full lifecycle on `demo.transactions`:
1. Create the table with hourly partitioning
2. Write 48 individual single-row appends across 6 hours → 48 tiny files
3. Delete a partition's worth of rows → Copy-on-Write removes those files
4. Compact with `rewrite_data_files` → dramatic file count reduction
5. Compact manifests with `rewrite_manifests`
6. Inspect the snapshot and file layout at each step

**Self-contained:** drops and recreates `demo.transactions` on every run.  
**Prerequisites:** `00_setup_catalog.ipynb` (S3 bucket + namespace only).

In [1]:
import datetime
import os

import polars as pl
from pyiceberg.catalog.rest import RestCatalog
from pyiceberg.exceptions import NoSuchTableError
from pyiceberg.expressions import LessThan
from pyiceberg.io.pyarrow import schema_to_pyarrow
from pyiceberg.partitioning import PartitionField, PartitionSpec
from pyiceberg.schema import Schema
from pyiceberg.transforms import HourTransform
from pyiceberg.types import (
    DoubleType,
    LongType,
    NestedField,
    StringType,
    TimestamptzType,
)

NESSIE_URI       = os.environ["NESSIE_URI"]
S3_ENDPOINT      = os.environ["AWS_S3_ENDPOINT"]
S3_ACCESS_KEY    = os.environ["AWS_ACCESS_KEY_ID"]
S3_SECRET_KEY    = os.environ["AWS_SECRET_ACCESS_KEY"]
WAREHOUSE_BUCKET = os.environ["ICEBERG_WAREHOUSE_BUCKET"]
WAREHOUSE_URI    = f"s3://{WAREHOUSE_BUCKET}/warehouse"

catalog = RestCatalog(
    name="nessie",
    **{
        "uri":                  NESSIE_URI,
        "warehouse":            WAREHOUSE_URI,
        "s3.endpoint":          S3_ENDPOINT,
        "s3.access-key-id":     S3_ACCESS_KEY,
        "s3.secret-access-key": S3_SECRET_KEY,
        "s3.path-style-access": "true",
        "s3.region":            "us-east-1",
    },
)
print("Connected to catalog:", catalog.name)

Connected to catalog: nessie


## Helpers

`file_summary()` is called four times (after write, after delete, after compaction,
and for the final check). Defining it once avoids repeating the same
`inspect.files()` + group-by logic in every section.

In [2]:
UTC = datetime.timezone.utc


def file_summary(tbl) -> pl.DataFrame:
    """Per-partition file count, record count, and size from inspect.files()."""
    tbl.refresh()
    files = pl.from_arrow(tbl.inspect.files())
    return (
        files.select(["file_path", "record_count", "file_size_in_bytes"])
        .with_columns(
            pl.col("file_path")
            .str.extract(r".*/data/([^/]+)/", 1)
            .alias("partition")
        )
        .group_by("partition")
        .agg(
            pl.len().alias("file_count"),
            pl.sum("record_count").alias("total_records"),
            pl.sum("file_size_in_bytes").alias("total_bytes"),
            pl.mean("file_size_in_bytes").cast(pl.Int64).alias("avg_bytes_per_file"),
        )
        .sort("partition", nulls_last=True)
    )


def print_snapshot(tbl) -> None:
    """Print the current snapshot's committed_at, operation, and id."""
    tbl.refresh()
    snaps = pl.from_arrow(tbl.inspect.snapshots())
    if len(snaps) == 0:
        print("  (no snapshot yet)")
        return
    print(snaps.select(["committed_at", "operation", "snapshot_id"]))

## 1. Create `demo.transactions`

Schema: `txn_id`, `account_id`, `merchant_id`, `amount`, `currency`, `txn_type`,
`created_at`. Partitioned by `hours(created_at)` — hourly granularity spreads 48
transactions across 6 partitions (8 rows each), giving us a controlled fragmentation
pattern that is large enough to demonstrate compaction but small enough to inspect.

The table is dropped and recreated on every run for reproducibility.

In [3]:
TABLE_ID = ("demo", "transactions")

iceberg_schema = Schema(
    NestedField(1, "txn_id",      StringType(),      required=True),
    NestedField(2, "account_id",  LongType(),        required=False),
    NestedField(3, "merchant_id", LongType(),        required=False),
    NestedField(4, "amount",      DoubleType(),      required=False),
    NestedField(5, "currency",    StringType(),      required=False),
    NestedField(6, "txn_type",    StringType(),      required=False),
    NestedField(7, "created_at",  TimestamptzType(), required=True),
)
partition_spec = PartitionSpec(
    PartitionField(source_id=7, field_id=1000, transform=HourTransform(), name="created_at_hour")
)

try:
    catalog.drop_table(TABLE_ID)
    print("Dropped existing table.")
except NoSuchTableError:
    print("No prior table — fresh create.")

table = catalog.create_table(
    identifier=TABLE_ID,
    schema=iceberg_schema,
    partition_spec=partition_spec,
)
arrow_schema = schema_to_pyarrow(iceberg_schema)
print("Table created:", table.name())
print("Location:     ", table.location())

Dropped existing table.
Table created: ('demo', 'transactions')
Location:      s3://iceberg-warehouse/warehouse/demo/transactions_a13afd7c-4175-4cd8-aff3-5bfae252ed85


## 2. Write 48 individual appends — extreme file fragmentation

Each `table.append()` call commits one Parquet file to S3. 48 calls → 48 files.
This simulates a payment processor emitting one transaction per micro-batch — a
common real-world pattern that causes rapid file proliferation.

**Layout:**
- 48 transactions across 6 hours: `2024-03-15 09:00` through `14:49 UTC`
- 8 transactions per hour → 8 files per partition × 6 partitions
- `amount` increases linearly (`10.0 + i × 2.5`), cycling currencies and txn types

In [4]:
BASE_DT    = datetime.datetime(2024, 3, 15, 9, 0, 0, tzinfo=UTC)
CURRENCIES = ["USD", "EUR", "GBP", "AZN", "TRY"]
TXN_TYPES  = ["purchase", "refund", "transfer", "fee"]

for i in range(48):
    hour_offset   = i // 8       # 0-5 → hours 09-14
    minute_offset = (i % 8) * 7  # 0, 7, 14, 21, 28, 35, 42, 49
    ts = BASE_DT + datetime.timedelta(hours=hour_offset, minutes=minute_offset)
    row = pl.DataFrame({
        "txn_id":      [f"txn{i:04d}"],
        "account_id":  [1000 + (i % 10)],
        "merchant_id": [500  + (i % 5)],
        "amount":      [round(10.0 + i * 2.5, 2)],
        "currency":    [CURRENCIES[i % len(CURRENCIES)]],
        "txn_type":    [TXN_TYPES[i % len(TXN_TYPES)]],
        "created_at":  [ts],
    })
    table.append(row.to_arrow().cast(arrow_schema))

table.refresh()
print(f"Total rows: {len(pl.from_arrow(table.scan().to_arrow()))}  (expected: 48)")

Total rows: 48  (expected: 48)


In [5]:
print("=== AFTER 48 APPENDS: File layout ===")
before_write = file_summary(table)
print(before_write)
print(f"\nTotal files  : {before_write['file_count'].sum()}  (expected: 48)")
print(f"Total bytes  : {before_write['total_bytes'].sum():,}")
print(f"Avg file size: {before_write['avg_bytes_per_file'].mean():.0f} bytes")
print("\nCurrent snapshot:")
print_snapshot(table)

=== AFTER 48 APPENDS: File layout ===


shape: (6, 5)
┌───────────────────────────────┬────────────┬───────────────┬─────────────┬────────────────────┐
│ partition                     ┆ file_count ┆ total_records ┆ total_bytes ┆ avg_bytes_per_file │
│ ---                           ┆ ---        ┆ ---           ┆ ---         ┆ ---                │
│ str                           ┆ u32        ┆ i64           ┆ i64         ┆ i64                │
╞═══════════════════════════════╪════════════╪═══════════════╪═════════════╪════════════════════╡
│ created_at_hour=2024-03-15-09 ┆ 8          ┆ 8             ┆ 27170       ┆ 3396               │
│ created_at_hour=2024-03-15-10 ┆ 8          ┆ 8             ┆ 27170       ┆ 3396               │
│ created_at_hour=2024-03-15-11 ┆ 8          ┆ 8             ┆ 27170       ┆ 3396               │
│ created_at_hour=2024-03-15-12 ┆ 8          ┆ 8             ┆ 27170       ┆ 3396               │
│ created_at_hour=2024-03-15-13 ┆ 8          ┆ 8             ┆ 27170       ┆ 3396               │
│ crea



Total files  : 48  (expected: 48)
Total bytes  : 163,020
Avg file size: 3396 bytes

Current snapshot:
shape: (1, 3)
┌─────────────────────────┬───────────┬─────────────────────┐
│ committed_at            ┆ operation ┆ snapshot_id         │
│ ---                     ┆ ---       ┆ ---                 │
│ datetime[ms]            ┆ str       ┆ i64                 │
╞═════════════════════════╪═══════════╪═════════════════════╡
│ 2026-05-15 14:40:07.129 ┆ append    ┆ 8316916665425432848 │
└─────────────────────────┴───────────┴─────────────────────┘


## 3. Delete rows — Copy-on-Write in action

`table.delete(delete_filter=LessThan("amount", 30.0))` targets transactions
`txn0000`–`txn0007` (amounts 10.0–27.5). These all land in the first partition
(`created_at_hour=2024-03-15-09`).

**What Copy-on-Write does:**  
With single-row Parquet files, every affected file contains exactly one row that
matches the predicate. PyIceberg reads each file, finds 0 surviving rows, and
removes the file reference from the manifest — no new file is written. The entire
hour-09 partition disappears from the current snapshot.

**Key insight:** deletion creates a new snapshot and changes the manifest, but it
does *not* compact the remaining 40 files in hours 10–14. Fragmentation persists
until you run compaction explicitly.

In [6]:
table.delete(delete_filter=LessThan("amount", 30.0))
table.refresh()
rows_after_delete = len(pl.from_arrow(table.scan().to_arrow()))
print(f"Rows after delete: {rows_after_delete}  (expected: 40)")
print("\nCurrent snapshot:")
print_snapshot(table)

Rows after delete: 40  (expected: 40)

Current snapshot:
shape: (1, 3)
┌─────────────────────────┬───────────┬─────────────────────┐
│ committed_at            ┆ operation ┆ snapshot_id         │
│ ---                     ┆ ---       ┆ ---                 │
│ datetime[ms]            ┆ str       ┆ i64                 │
╞═════════════════════════╪═══════════╪═════════════════════╡
│ 2026-05-15 14:40:07.976 ┆ delete    ┆ 4567649937550655902 │
└─────────────────────────┴───────────┴─────────────────────┘


In [7]:
print("=== AFTER DELETE: File layout ===")
after_delete = file_summary(table)
print(after_delete)
print(f"\nTotal files: {after_delete['file_count'].sum()}  (expected: 40)")
print("(hour-09 partition is absent — all 8 rows were deleted)")

=== AFTER DELETE: File layout ===
shape: (5, 5)
┌───────────────────────────────┬────────────┬───────────────┬─────────────┬────────────────────┐
│ partition                     ┆ file_count ┆ total_records ┆ total_bytes ┆ avg_bytes_per_file │
│ ---                           ┆ ---        ┆ ---           ┆ ---         ┆ ---                │
│ str                           ┆ u32        ┆ i64           ┆ i64         ┆ i64                │
╞═══════════════════════════════╪════════════╪═══════════════╪═════════════╪════════════════════╡
│ created_at_hour=2024-03-15-10 ┆ 8          ┆ 8             ┆ 27170       ┆ 3396               │
│ created_at_hour=2024-03-15-11 ┆ 8          ┆ 8             ┆ 27170       ┆ 3396               │
│ created_at_hour=2024-03-15-12 ┆ 8          ┆ 8             ┆ 27170       ┆ 3396               │
│ created_at_hour=2024-03-15-13 ┆ 8          ┆ 8             ┆ 27170       ┆ 3396               │
│ created_at_hour=2024-03-15-14 ┆ 8          ┆ 8             ┆ 27170  

## 4. Compact data files — `rewrite_data_files`

Start PySpark in `local[*]` mode connecting to the same Nessie catalog.
The stored procedure `system.rewrite_data_files` uses **bin-packing** by default:

1. Groups files by partition
2. Within each group, collects files smaller than `target-file-size-bytes / 2`
   (i.e., below 512 KB with our 1 MB target)
3. Merges qualifying groups that have at least `min-input-files` files
4. Writes new right-sized Parquet files and atomically replaces old ones

Our single-row files (~2.5 KB each) are well below the 512 KB threshold, so all
8-file partitions are eligible. Setting `min-input-files=2` overrides the default
(5) so every remaining partition participates.

**Expected result:** hours 10–14 (8 files each) → 1 file each → 5 files total.

In [8]:
from pyspark.sql import SparkSession

PACKAGES = ",".join([
    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1",
    "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.99.0",
    "software.amazon.awssdk:bundle:2.24.0",
    # hadoop-aws gives Spark the s3a filesystem for S3 scans.
    "org.apache.hadoop:hadoop-aws:3.3.4",
])

spark = (
    SparkSession.builder
    .appName("lakehouse-compaction-demo")
    .master("local[*]")
    .config("spark.jars.packages", PACKAGES)
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
        "org.projectnessie.spark.extensions.NessieSparkSessionExtensions",
    )
    .config("spark.sql.catalog.nessie",             "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .config("spark.sql.catalog.nessie.uri",          "http://nessie:19120/api/v2")
    .config("spark.sql.catalog.nessie.ref",          "main")
    .config("spark.sql.catalog.nessie.warehouse",    f"s3://{WAREHOUSE_BUCKET}/warehouse")
    .config("spark.sql.catalog.nessie.io-impl",      "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.nessie.s3.endpoint",          S3_ENDPOINT)
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true")
    .config("spark.sql.catalog.nessie.s3.access-key-id",     S3_ACCESS_KEY)
    .config("spark.sql.catalog.nessie.s3.secret-access-key", S3_SECRET_KEY)
    # Map both s3 and s3a schemes to S3AFileSystem.
    .config("spark.hadoop.fs.s3.impl",  "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.endpoint",           S3_ENDPOINT)
    .config("spark.hadoop.fs.s3a.path.style.access",  "true")
    .config("spark.hadoop.fs.s3a.access.key",         S3_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",         S3_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.jars.ivy", "/home/jovyan/.ivy2")
    .getOrCreate()
)
print("Spark version:", spark.version)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12 added as a dependency
software.amazon.awssdk#bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8e0ed53d-0427-45a0-9697-c7a0ab34cbb4;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
	found org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12;0.99.0 in central
	found software.amazon.awssdk#bundle;2.24.0 in central


	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 244ms :: artifacts dl 6ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 from central in [default]
	org.projectnessie.nessie-integrations#nessie-spark-extensions-3.5_2.12;0.99.0 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	software.amazon.awssdk#bundle;2.24.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark version: 3.5.3


In [9]:
spark.sql("SHOW TABLES IN nessie.demo").show()

+---------+------------+-----------+
|namespace|   tableName|isTemporary|
+---------+------------+-----------+
|     demo|      events|      false|
|     demo|transactions|      false|
+---------+------------+-----------+



### 4a. Run `rewrite_data_files`

Result columns:
- `rewritten_data_files_count` — old files consumed
- `added_data_files_count` — new consolidated files written
- `rewritten_bytes_count` — bytes processed
- `failed_data_files_count` — should be 0

In [10]:
print("Running rewrite_data_files ...")
spark.sql("""
    CALL nessie.system.rewrite_data_files(
      table   => 'demo.transactions',
      options => map(
        'min-input-files',       '2',
        'target-file-size-bytes','1048576'
      )
    )
""").show(truncate=False)

Running rewrite_data_files ...


SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|40                        |5                     |135850               |0                      |
+--------------------------+----------------------+---------------------+-----------------------+



In [11]:
print("=== AFTER rewrite_data_files: File layout ===")
after_compact = file_summary(table)
print(after_compact)
print(f"\nTotal files: {after_compact['file_count'].sum()}  (expected: 5)")
rows_after_compact = len(pl.from_arrow(table.scan().to_arrow()))
print(f"Row count  : {rows_after_compact}  (must equal {rows_after_delete})")
print("\nCurrent snapshot:")
print_snapshot(table)

=== AFTER rewrite_data_files: File layout ===


shape: (5, 5)
┌───────────────────────────────┬────────────┬───────────────┬─────────────┬────────────────────┐
│ partition                     ┆ file_count ┆ total_records ┆ total_bytes ┆ avg_bytes_per_file │
│ ---                           ┆ ---        ┆ ---           ┆ ---         ┆ ---                │
│ str                           ┆ u32        ┆ i64           ┆ i64         ┆ i64                │
╞═══════════════════════════════╪════════════╪═══════════════╪═════════════╪════════════════════╡
│ created_at_hour=2024-03-15-10 ┆ 1          ┆ 8             ┆ 2383        ┆ 2383               │
│ created_at_hour=2024-03-15-11 ┆ 1          ┆ 8             ┆ 2388        ┆ 2388               │
│ created_at_hour=2024-03-15-12 ┆ 1          ┆ 8             ┆ 2388        ┆ 2388               │
│ created_at_hour=2024-03-15-13 ┆ 1          ┆ 8             ┆ 2388        ┆ 2388               │
│ created_at_hour=2024-03-15-14 ┆ 1          ┆ 8             ┆ 2385        ┆ 2385               │
└─────

### 4b. Before / after comparison

Three snapshots of the file layout joined by partition:

In [12]:
comparison = (
    before_write.select(["partition", "file_count"])
    .rename({"file_count": "after_write"})
    .join(
        after_delete.select(["partition", "file_count"]).rename({"file_count": "after_delete"}),
        on="partition", how="full", coalesce=True,
    )
    .join(
        after_compact.select(["partition", "file_count"]).rename({"file_count": "after_compact"}),
        on="partition", how="full", coalesce=True,
    )
    .fill_null(0)
    .sort("partition", nulls_last=True)
)
print(comparison)

total_write   = int(before_write["file_count"].sum())
total_delete  = int(after_delete["file_count"].sum())
total_compact = int(after_compact["file_count"].sum())
print(f"\nFile trajectory : {total_write} → {total_delete} → {total_compact}")
print(f"Peak → compacted: {total_write} → {total_compact}  "
      f"({(1 - total_compact / total_write) * 100:.0f}% fewer files)")

shape: (6, 4)
┌───────────────────────────────┬─────────────┬──────────────┬───────────────┐
│ partition                     ┆ after_write ┆ after_delete ┆ after_compact │
│ ---                           ┆ ---         ┆ ---          ┆ ---           │
│ str                           ┆ u32         ┆ u32          ┆ u32           │
╞═══════════════════════════════╪═════════════╪══════════════╪═══════════════╡
│ created_at_hour=2024-03-15-09 ┆ 8           ┆ 0            ┆ 0             │
│ created_at_hour=2024-03-15-10 ┆ 8           ┆ 8            ┆ 1             │
│ created_at_hour=2024-03-15-11 ┆ 8           ┆ 8            ┆ 1             │
│ created_at_hour=2024-03-15-12 ┆ 8           ┆ 8            ┆ 1             │
│ created_at_hour=2024-03-15-13 ┆ 8           ┆ 8            ┆ 1             │
│ created_at_hour=2024-03-15-14 ┆ 8           ┆ 8            ┆ 1             │
└───────────────────────────────┴─────────────┴──────────────┴───────────────┘

File trajectory : 48 → 40 → 5
Peak → 

## 5. Rewrite manifests — `rewrite_manifests`

After 48 appends, 1 delete, and 1 `rewrite_data_files`, the manifest layer has
accumulated multiple manifest files — one is created per snapshot. The
`rewrite_manifests` procedure consolidates them into a minimal set.

**Why it matters:** a table with many manifests forces every query planner to open
and deserialise each one to determine which data files to scan. Fewer manifests =
faster planning, especially on large tables.

Result columns: `rewritten_manifests_count`, `added_manifests_count`.

In [13]:
table.refresh()
manifests_before = pl.from_arrow(table.inspect.manifests())
print(f"Manifests before rewrite: {len(manifests_before)}")
print(manifests_before.select(["path", "length", "added_data_files_count", "existing_data_files_count"]))

Manifests before rewrite: 41
shape: (41, 4)
┌─────────────────────────────────┬────────┬────────────────────────┬───────────────────────────┐
│ path                            ┆ length ┆ added_data_files_count ┆ existing_data_files_count │
│ ---                             ┆ ---    ┆ ---                    ┆ ---                       │
│ str                             ┆ i64    ┆ i32                    ┆ i32                       │
╞═════════════════════════════════╪════════╪════════════════════════╪═══════════════════════════╡
│ s3://iceberg-warehouse/warehou… ┆ 7709   ┆ 5                      ┆ 0                         │
│ s3://iceberg-warehouse/warehou… ┆ 7345   ┆ 0                      ┆ 0                         │
│ s3://iceberg-warehouse/warehou… ┆ 7345   ┆ 0                      ┆ 0                         │
│ s3://iceberg-warehouse/warehou… ┆ 7347   ┆ 0                      ┆ 0                         │
│ s3://iceberg-warehouse/warehou… ┆ 7348   ┆ 0                      ┆ 0   

In [14]:
print("Running rewrite_manifests ...")
spark.sql("""
    CALL nessie.system.rewrite_manifests(
      table => 'demo.transactions'
    )
""").show(truncate=False)

Running rewrite_manifests ...


26/05/15 14:40:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------------------+---------------------+
|rewritten_manifests_count|added_manifests_count|
+-------------------------+---------------------+
|41                       |1                    |
+-------------------------+---------------------+



In [15]:
table.refresh()
manifests_after = pl.from_arrow(table.inspect.manifests())
print(f"Manifests after rewrite : {len(manifests_after)}")
print(f"Reduction               : {len(manifests_before)} → {len(manifests_after)}")

Manifests after rewrite : 1
Reduction               : 41 → 1


## 6. Snapshot timeline

Each operation produced a new Iceberg snapshot:
- 48 `append` snapshots (one per row written in the loop)
- 1 `delete` snapshot (the CoW delete)
- 1 `replace` snapshot (`rewrite_data_files`)
- 1 `replace` snapshot (`rewrite_manifests`)

**Nessie caveat:** `inspect.snapshots()` on a Nessie-backed table returns only the
snapshot referenced by the current Nessie commit — typically 1 row. The 48
intermediate append snapshots each existed briefly but are replaced by the next
commit. The full history lives in the Nessie commit log (see notebook 08–09).

In [16]:
table.refresh()
print("Snapshots visible to PyIceberg:")
print(
    pl.from_arrow(table.inspect.snapshots())
    .select(["committed_at", "operation", "snapshot_id"])
)
print("\nHistory entries:")
print(pl.from_arrow(table.inspect.history()))

Snapshots visible to PyIceberg:
shape: (1, 3)
┌─────────────────────────┬───────────┬─────────────────────┐
│ committed_at            ┆ operation ┆ snapshot_id         │
│ ---                     ┆ ---       ┆ ---                 │
│ datetime[ms]            ┆ str       ┆ i64                 │
╞═════════════════════════╪═══════════╪═════════════════════╡
│ 2026-05-15 14:40:28.664 ┆ replace   ┆ 3618615205240344895 │
└─────────────────────────┴───────────┴─────────────────────┘

History entries:
shape: (1, 4)
┌─────────────────────────┬─────────────────────┬───────────┬─────────────────────┐
│ made_current_at         ┆ snapshot_id         ┆ parent_id ┆ is_current_ancestor │
│ ---                     ┆ ---                 ┆ ---       ┆ ---                 │
│ datetime[ms]            ┆ i64                 ┆ i64       ┆ bool                │
╞═════════════════════════╪═════════════════════╪═══════════╪═════════════════════╡
│ 2026-05-15 14:40:28.664 ┆ 3618615205240344895 ┆ null      ┆ true  

## 7. Final verification — logical content is unchanged

Compaction and manifest rewrite are purely physical operations. They never modify
query results — only how the data is stored on disk and described in metadata.

In [17]:
df_final = pl.from_arrow(table.scan().to_arrow()).sort("created_at")
print(f"Final row count: {len(df_final)}  (expected: 40)")
print("\nFirst 5 rows:")
print(df_final.head(5))
print("\nLast 5 rows:")
print(df_final.tail(5))

Final row count: 40  (expected: 40)

First 5 rows:
shape: (5, 7)
┌─────────┬────────────┬─────────────┬────────┬──────────┬──────────┬─────────────────────────┐
│ txn_id  ┆ account_id ┆ merchant_id ┆ amount ┆ currency ┆ txn_type ┆ created_at              │
│ ---     ┆ ---        ┆ ---         ┆ ---    ┆ ---      ┆ ---      ┆ ---                     │
│ str     ┆ i64        ┆ i64         ┆ f64    ┆ str      ┆ str      ┆ datetime[μs, UTC]       │
╞═════════╪════════════╪═════════════╪════════╪══════════╪══════════╪═════════════════════════╡
│ txn0008 ┆ 1008       ┆ 503         ┆ 30.0   ┆ AZN      ┆ purchase ┆ 2024-03-15 10:00:00 UTC │
│ txn0009 ┆ 1009       ┆ 504         ┆ 32.5   ┆ TRY      ┆ refund   ┆ 2024-03-15 10:07:00 UTC │
│ txn0010 ┆ 1000       ┆ 500         ┆ 35.0   ┆ USD      ┆ transfer ┆ 2024-03-15 10:14:00 UTC │
│ txn0011 ┆ 1001       ┆ 501         ┆ 37.5   ┆ EUR      ┆ fee      ┆ 2024-03-15 10:21:00 UTC │
│ txn0012 ┆ 1002       ┆ 502         ┆ 40.0   ┆ GBP      ┆ purchase ┆ 2

In [18]:
print("=== FINAL file layout ===")
final = file_summary(table)
print(final)
total_f = int(final["file_count"].sum())
total_b = int(final["total_bytes"].sum())
print(f"\nTotal files : {total_f}")
print(f"Total bytes : {total_b:,}")
print(f"Avg per file: {total_b // total_f:,} bytes")

=== FINAL file layout ===
shape: (5, 5)
┌───────────────────────────────┬────────────┬───────────────┬─────────────┬────────────────────┐
│ partition                     ┆ file_count ┆ total_records ┆ total_bytes ┆ avg_bytes_per_file │
│ ---                           ┆ ---        ┆ ---           ┆ ---         ┆ ---                │
│ str                           ┆ u32        ┆ i64           ┆ i64         ┆ i64                │
╞═══════════════════════════════╪════════════╪═══════════════╪═════════════╪════════════════════╡
│ created_at_hour=2024-03-15-10 ┆ 1          ┆ 8             ┆ 2383        ┆ 2383               │
│ created_at_hour=2024-03-15-11 ┆ 1          ┆ 8             ┆ 2388        ┆ 2388               │
│ created_at_hour=2024-03-15-12 ┆ 1          ┆ 8             ┆ 2388        ┆ 2388               │
│ created_at_hour=2024-03-15-13 ┆ 1          ┆ 8             ┆ 2388        ┆ 2388               │
│ created_at_hour=2024-03-15-14 ┆ 1          ┆ 8             ┆ 2385        ┆ 2

In [19]:
spark.stop()
print("SparkSession stopped.")

SparkSession stopped.


---

## Summary

| Step | Operation | Files | Rows |
|------|-----------|------:|-----:|
| Create | — | 0 | 0 |
| 48 appends | `append` ×48 | 48 | 48 |
| Delete | `LessThan("amount", 30.0)` | 40 | 40 |
| `rewrite_data_files` | Spark stored procedure | 5 | 40 |
| `rewrite_manifests` | Spark stored procedure | 5 | 40 |

**90% file reduction** — from 48 tiny files to 5 right-sized files — with no
change to the logical table content.

### Production recommendations

- Schedule `rewrite_data_files` on tables receiving streaming ingest — daily or
  triggered when per-partition file count exceeds a threshold (e.g., > 100).
- Use `min-input-files=5` (the default) and `target-file-size-bytes=134217728`
  (128 MB) in production. Only lower `min-input-files` if you have specific
  partitions with known 2–4 file fragmentation.
- Run `rewrite_manifests` in the same maintenance window as `rewrite_data_files`.
- **On Nessie, use [`nessie-gc`](https://projectnessie.org/nessie-latest/gc/)
  instead of `remove_orphan_files` for garbage collection.** Spark's
  `remove_orphan_files` is unaware of Nessie refs and will mark files referenced
  by other branches/history as orphans, corrupting them. See notebook 10 for the
  full warning.

See [docs/iceberg/CURRICULUM.md](../docs/iceberg/CURRICULUM.md) for the notebook
index and dependency DAG.